<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/12_SVM_Hub_Carrier_Markets_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12. Support Vector Machine: Drawing the Boundary

Notebook 9 asked which markets are led by American, Delta or United, and answered by looking at neighbors.
A **support vector machine (SVM)** answers the same question by drawing a **boundary** between the two groups,
placed so that the gap (the **margin**) between the boundary and the nearest markets on each side is as wide as
possible. Those nearest markets are the **support vectors**: they alone decide where the boundary goes.

Slides: W3.1, support vector machine, kernel mapping, kernel functions.

> **Data.** The 200 busiest U.S. domestic markets in 2019 (both directions combined), read straight from the course
> GitHub repository. Source: Sabre Market Intelligence, licensed to Embry-Riddle.

> **Figures.** The figures in this notebook are pictures made by its full version (`12_SVM_Hub_Carrier_Markets_all.ipynb`), so you can focus on the code that fits the model and produces the results.

## 1. Data: Two Inputs, So We Can See the Boundary

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay

routes = pd.read_csv('https://raw.githubusercontent.com/kjmobile/lb2/main/data/sabre_routes_2019.csv')
# 1 = led by a big network airline (American, Delta or United), 0 = led by another airline
routes['hub_led'] = routes['top_airline'].isin(['AA', 'DL', 'UA']).astype(int)

X = routes[['lcc_share', 'hhi']]       # both between 0 and 1, so no scaling needed here
y = routes['hub_led']
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=320, stratify=y)
print('train:', train_X.shape, ' test:', test_X.shape)
print('baseline (always guess the larger group):', round(test_y.mean(), 2))

train: (150, 2)  test: (50, 2)
baseline (always guess the larger group): 0.58


`lcc_share` is the share of passengers on low-cost airlines; `hhi` is market concentration (1 = one airline
carries everyone).

## 2. A Straight Boundary: Linear SVM

In [2]:
linear_svm = SVC(kernel='linear', C=1)
linear_svm.fit(train_X, train_y)

print('train accuracy:', round(linear_svm.score(train_X, train_y), 3))
print('test accuracy: ', round(linear_svm.score(test_X, test_y), 3))
print('support vectors:', linear_svm.n_support_.sum(), 'of', len(train_X), 'training markets')

train accuracy: 0.807
test accuracy:  0.86
support vectors: 100 of 150 training markets


**Figure 1**

![Figure 1](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/12_fig1.png)

The boundary is a straight line, $w_1 \cdot \text{lcc\_share} + w_2 \cdot \text{hhi} + b = 0$:

In [3]:
print('weights (lcc_share, hhi):', linear_svm.coef_.round(2))
print('b:', linear_svm.intercept_.round(2))

weights (lcc_share, hhi): [[-3.1  -0.37]]
b: [1.85]


The weight on low-cost share is large and the weight on concentration is small, so the line is nearly vertical:
markets with a low-cost share below about 0.55 are predicted to be led by American, Delta or United.

Why are so many markets support vectors? The two groups overlap, and `C=1` allows a wide, **soft margin** that
tolerates mistakes. Every market inside the margin, or on the wrong side, counts as a support vector.

## 3. A Curved Boundary: The Kernel Trick

When a straight line cannot separate the groups, a **kernel** maps the data into a higher-dimensional space where a
flat boundary can. Back in the original two inputs, that boundary looks curved. The **RBF** (radial basis function)
kernel measures similarity by closeness, like KNN, and is the most common choice.

In [4]:
rbf_svm = SVC(kernel='rbf', C=1)
rbf_svm.fit(train_X, train_y)

print('train accuracy:', round(rbf_svm.score(train_X, train_y), 3))
print('test accuracy: ', round(rbf_svm.score(test_X, test_y), 3))
print('support vectors:', rbf_svm.n_support_.sum())

train accuracy: 0.813
test accuracy:  0.86
support vectors: 76


**Figure 2**

![Figure 2](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/12_fig2.png)

The curved boundary bends in two places. Higher up, in more concentrated markets, it allows less low-cost share:
a market that one airline dominates and that has even a modest low-cost share is usually led by Southwest or another
airline. And in the bottom-left corner (almost no low-cost passengers, several competitors) it predicts "other
airline": markets such as HNL-LAX and ORD-SEA, led by Hawaiian and Alaska. On the 50 test markets the two kernels score the same; one market is 2 percentage points, so small
differences here are luck.

## 4. Compare with Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression().fit(train_X, train_y)
print('logistic regression, test accuracy:', round(logit.score(test_X, test_y), 3))

logistic regression, test accuracy: 0.84


## 5. The C Setting

`C` sets the price of a mistake on the training markets. A small `C` keeps a wide margin and tolerates mistakes;
a large `C` bends the boundary to classify every training market correctly, with the risk of overfitting.

### Your turn

Fit the RBF SVM with **C = 100** and compare it with C = 1 above.

1. In the cell below, fill in the blanks (`___`) and run it.
2. Then finish this sentence:
   *With C = 100, training accuracy rises to ___ (from 0.813), test accuracy is ___ (from 0.86), and the model
   keeps ___ support vectors (from 76).*

<details><summary>Show answer</summary>

```python
rbf_100 = SVC(kernel='rbf', C=100)
rbf_100.fit(train_X, train_y)

print('train accuracy:', round(rbf_100.score(train_X, train_y), 3))
print('test accuracy: ', round(rbf_100.score(test_X, test_y), 3))
print('support vectors:', rbf_100.n_support_.sum())
```

*With C = 100, training accuracy rises to **0.867** (from 0.813), test accuracy is **0.84** (from 0.86), and the model keeps **62** support vectors (from 76). A harder margin fits the training markets better but not new ones.*
</details>

In [ ]:
rbf_100 = SVC(kernel='___', C=___)
rbf_100.fit(train_X, train_y)

print('train accuracy:', round(rbf_100.score(train_X, train_y), 3))
print('test accuracy: ', round(rbf_100.score(test_X, test_y), 3))
print('support vectors:', rbf_100.n_support_.sum())

## 6. Summary

| Model | Test accuracy (50 markets) | Note |
|---|---|---|
| Always guess the larger group | 0.58 | |
| Linear SVM, C = 1 | 0.86 | boundary near a low-cost share of 0.55 |
| RBF SVM, C = 1 | 0.86 | curved boundary |
| Logistic regression | 0.84 | also a straight boundary |
| KNN, k = 5 (notebook 9, three inputs) | 0.86 | neighbors instead of a boundary |

---
# Challenge

1. Add `distance_mi` as a third input. Now standardization matters (miles vs. shares). Fit the RBF SVM with and
   without `StandardScaler` and compare.
2. Try `kernel='poly'` (a polynomial kernel, `degree=3`). How does the boundary look?
3. The linear SVM says concentration barely matters. Look at the scatter plot: do you agree?

In [7]:
# Your answer